In [1]:
# Install Python packages
%pip install anthropic python-dotenv

# Load env variable
from dotenv import load_dotenv
import os

load_dotenv()

# Create an API Client
from anthropic import Anthropic

client = Anthropic(
    api_key=os.getenv("ANTHROPIC_API_KEY"),
    base_url=os.getenv("ANTHROPIC_BASE_URI")
)
model = "claude-sonnet-4-6"

# Making a Helper Fxn
def add_user_message(messages, text):
    user_message = {"role":"user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role":"assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system = None, temperature = 1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    
    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"]=stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

Note: you may need to restart the kernel to use updated packages.


In [2]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
# add_assistant_message(messages, "```json")

raw_text = chat(messages, stop_sequences=["```\n\n"])

clean_json=raw_text.split("```json")[-1]

clean_json


'\n{\n  "Name": "my-rule",\n  "EventPattern": {\n    "source": ["aws.ec2"],\n    "detail-type": ["EC2 Instance State-change Notification"],\n    "detail": {\n      "state": ["stopped"]\n    }\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Id": "my-target",\n      "Arn": "arn:aws:sns:us-east-1:123456789012:my-topic"\n    }\n  ]\n}\n'

In [3]:
import json

json.loads(clean_json.strip())

{'Name': 'my-rule',
 'EventPattern': {'source': ['aws.ec2'],
  'detail-type': ['EC2 Instance State-change Notification'],
  'detail': {'state': ['stopped']}},
 'State': 'ENABLED',
 'Targets': [{'Id': 'my-target',
   'Arn': 'arn:aws:sns:us-east-1:123456789012:my-topic'}]}